# KNN Artifact Detection

**Dataset**: PhysioNet Auditory EEG  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

We use K-Nearest Neighbors (KNN) to detect artifacts in EEG. We extract statistical features from time windows and train the model to distinguish clean from artifact windows.

## Expected outputs

- Ground truth artifacts (threshold-based) on top
- KNN predictions on bottom with accuracy score
- Red shaded regions indicate detected artifacts

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| Channel | P4 | Parietal region |
| WINDOW_SIZE | 200 | One second window |
| n_neighbors | 5 | Number of neighbors |
| threshold | 3*std | Artifact threshold |


## 1. Install dependencies


In [ ]:
!pip install mne scikit-learn EMD-signal scipy numpy plotly wfdb


## 2. Clone repo and download data

We download only subject 1 (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2.


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


## 4. Apply KNN for artifact detection

We extract features (variance, kurtosis, max amplitude) from each window, label them based on amplitude threshold, then train KNN.


In [ ]:
from scipy.stats import kurtosis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import accuracy_score

channel_data = eeg_data[:, 0]
WINDOW_SIZE = 200
n_windows = len(channel_data) // WINDOW_SIZE
windows = channel_data[:n_windows * WINDOW_SIZE].reshape(n_windows, WINDOW_SIZE)

features = np.column_stack([
    np.var(windows, axis=1),
    kurtosis(windows, axis=1),
    np.max(np.abs(windows), axis=1),
])

threshold = 3 * np.std(channel_data)
labels = (np.max(np.abs(windows), axis=1) > threshold).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.3, random_state=42
)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
predictions = cross_val_predict(KNeighborsClassifier(n_neighbors=5), features, labels, cv=5)
accuracy = accuracy_score(y_test, knn.predict(X_test))
print(f'Accuracy: {accuracy:.1%}')

## 5. Interactive plot

**What to look for:**

- Red shaded regions indicate artifacts
- Top: ground truth, bottom: KNN predictions
- Zoom in to inspect specific time ranges



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

t_sec = np.arange(len(channel_data)) / fs

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=(f'Ground truth (threshold={threshold:.0f} uV)',
                                    f'KNN predictions (accuracy={accuracy:.1%})'))
fig.add_trace(go.Scatter(x=t_sec, y=channel_data, name='Signal',
                         line=dict(color='blue', width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=channel_data, name='Signal',
                         line=dict(color='blue', width=0.5)), row=2, col=1)

for i in range(n_windows):
    t_start = i * WINDOW_SIZE / fs
    t_end = (i + 1) * WINDOW_SIZE / fs
    if labels[i] == 1:
        fig.add_vrect(x0=t_start, x1=t_end, fillcolor='red', opacity=0.3,
                      line_width=0, row=1, col=1)
    if predictions[i] == 1:
        fig.add_vrect(x0=t_start, x1=t_end, fillcolor='red', opacity=0.3,
                      line_width=0, row=2, col=1)

fig.update_layout(height=700, title_text='KNN Artifact Detection - Channel P4',
                  xaxis2_title='Time (s)', showlegend=False)
fig.show()


## What did we learn?

- KNN classifies windows based on similarity to neighbors
- Statistical features suffice to detect high-amplitude artifacts
- Label quality determines model quality
- Requires expert manual labeling in practice

